# SentinelVision AI
## Notebook 04 — Baseline Anomaly Models

### Objective

Notebook 03 created reusable clip embeddings from real UCF-Crime video clips.

This notebook trains the first anomaly detection models using those embeddings.

We will build:

1. a simple distance-threshold baseline,
2. an Isolation Forest model.

The goal is to establish a clean baseline before moving to deep learning.

The flow is:

`clip embeddings → train on normal training clips → anomaly scores → validation threshold → test evaluation`

In [1]:
from pathlib import Path

import joblib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from sklearn.ensemble import IsolationForest
from sklearn.metrics import (
    average_precision_score,
    classification_report,
    confusion_matrix,
    f1_score,
    precision_recall_curve,
    roc_auc_score,
)
from sklearn.preprocessing import StandardScaler

#### Load embeddings

In [2]:
embeddings_path = Path("../data/embeddings/clip_embeddings_sample.parquet")

embeddings = pd.read_parquet(embeddings_path)

embeddings.shape

(498, 23)

In [3]:
embeddings.head()

,clip_id,video_id,category,binary_label,split,start_seconds,end_seconds,feature_000,feature_001,feature_002,...,feature_006,feature_007,feature_008,feature_009,feature_010,feature_011,feature_012,feature_013,feature_014,feature_015
0,Shooting022_x264_clip_00010,Shooting022_x264,Shooting,1,test,40.0,44.0,0.416163,0.386835,0.374141,...,0.394131,0.225522,0.000269,0.000158,0.000236,0.000118,0.000140,0.000204,0.000162,0.000087
1,Vandalism041_x264_clip_00021,Vandalism041_x264,Vandalism,1,test,84.0,88.0,0.257803,0.171437,0.062419,...,0.184771,0.186079,0.002649,0.001979,0.000456,0.003085,0.002011,0.000281,0.001935,0.001966
2,Robbery130_x264_clip_00002,Robbery130_x264,Robbery,1,test,8.0,12.0,0.425955,0.425692,0.420834,...,0.425198,0.173660,0.003575,0.002963,0.002644,0.006365,0.006437,0.005010,0.003006,0.006305
3,Normal_Videos_641_x264_clip_00025,Normal_Videos_641_x264,Normal,0,test,100.0,104.0,0.289380,0.271774,0.224172,...,0.271585,0.246034,0.007098,0.005536,0.003933,0.000932,0.002951,0.001615,0.005603,0.002558
4,Abuse015_x264_clip_00012,Abuse015_x264,Abuse,1,test,48.0,52.0,0.503250,0.506105,0.498351,...,0.504288,0.403972,0.000653,0.000701,0.000608,0.000398,0.000426,0.000355,0.000664,0.000405


#### Identify feature columns

In [4]:
feature_columns = [
    column
    for column in embeddings.columns
    if column.startswith("feature_")
]

len(feature_columns), feature_columns[:5]

(16,
 ['feature_000', 'feature_001', 'feature_002', 'feature_003', 'feature_004'])

In [5]:
metadata_columns = [
    column
    for column in embeddings.columns
    if column not in feature_columns
]

metadata_columns

['clip_id',
 'video_id',
 'category',
 'binary_label',
 'split',
 'start_seconds',
 'end_seconds']

#### Split embeddings